# Item 8 — Pipeline ETL com Apache Spark (PySpark)

Pipeline de transformação e qualidade sobre o dataset `olist_order_items_dataset`,
implementando as camadas **Bronze → Silver → Gold** do Data Warehouse proposto no Item 6.

| Camada | Descrição |
|---|---|
| Bronze | Ingestão do CSV bruto |
| Silver | Limpeza, tipagem e validações de qualidade |
| Gold | Star Schema: dims + fato + views analíticas |

## 1. Instalação e Imports

In [ ]:
!pip install pyspark --quiet

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, TimestampType
from pyspark.sql.window import Window
import hashlib

## 2. Criação da SparkSession

In [ ]:
spark = SparkSession.builder \
    .appName("ETL_Olist_Ecommerce") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

## 3. Camada Bronze — Ingestão do CSV bruto

In [ ]:
arquivo = "/content/olist_order_items_dataset.csv"

df_bronze = spark.read.csv(
    arquivo,
    header=True,
    inferSchema=True
)

print(f"Bronze — linhas: {df_bronze.count()} | colunas: {len(df_bronze.columns)}")
df_bronze.printSchema()
df_bronze.show(5, truncate=False)

## 4. Camada Silver — Limpeza, tipagem e qualidade

Regras aplicadas (mesmas Expectations do Item 4):
- Colunas-chave não podem ser nulas
- `price` > 0
- `freight_value` >= 0
- Deduplicação por `order_id + order_item_id`

In [ ]:
# Padroniza nomes de colunas para maiúsculas
df = df_bronze
for col in df.columns:
    df = df.withColumnRenamed(col, col.upper())

# Tipagem explícita
df_silver = df \
    .withColumn("PRICE",              F.col("PRICE").cast(DoubleType())) \
    .withColumn("FREIGHT_VALUE",      F.col("FREIGHT_VALUE").cast(DoubleType())) \
    .withColumn("ORDER_ITEM_ID",      F.col("ORDER_ITEM_ID").cast(IntegerType())) \
    .withColumn("SHIPPING_LIMIT_DATE",F.col("SHIPPING_LIMIT_DATE").cast(TimestampType())) \
    .filter(F.col("ORDER_ID").isNotNull()) \
    .filter(F.col("PRODUCT_ID").isNotNull()) \
    .filter(F.col("SELLER_ID").isNotNull()) \
    .filter(F.col("PRICE") > 0) \
    .filter(F.col("FREIGHT_VALUE") >= 0) \
    .dropDuplicates(["ORDER_ID", "ORDER_ITEM_ID"])

print(f"Silver — linhas: {df_silver.count()}")
df_silver.show(5, truncate=False)

### 4.1 Relatório de qualidade da camada Silver

In [ ]:
total = df_silver.count()
removidos = df_bronze.count() - total

relatorio = {
    "total_linhas_bronze":  df_bronze.count(),
    "total_linhas_silver":  total,
    "linhas_removidas":     removidos,
    "nulos_order_id":       df_silver.filter(F.col("ORDER_ID").isNull()).count(),
    "nulos_product_id":     df_silver.filter(F.col("PRODUCT_ID").isNull()).count(),
    "price_invalido":       df_silver.filter(F.col("PRICE") <= 0).count(),
    "frete_invalido":       df_silver.filter(F.col("FREIGHT_VALUE") < 0).count(),
}

import json
print(json.dumps(relatorio, indent=2))

## 5. Camada Gold — Star Schema com Spark SQL

Implementação do modelo proposto no Item 6 (Kimball Star Schema).

In [ ]:
# Registra a camada Silver como tabela temporária
df_silver.createOrReplaceTempView("olist_silver")

In [ ]:
# dim_produto
dim_produto = spark.sql("""
    SELECT DISTINCT
        MD5(PRODUCT_ID)  AS product_key,
        PRODUCT_ID       AS product_id
    FROM olist_silver
""")
dim_produto.createOrReplaceTempView("dim_produto")
print(f"dim_produto: {dim_produto.count()} produtos")
dim_produto.show(5)

In [ ]:
# dim_vendedor
dim_vendedor = spark.sql("""
    SELECT DISTINCT
        MD5(SELLER_ID)   AS seller_key,
        SELLER_ID        AS seller_id
    FROM olist_silver
""")
dim_vendedor.createOrReplaceTempView("dim_vendedor")
print(f"dim_vendedor: {dim_vendedor.count()} vendedores")
dim_vendedor.show(5)

In [ ]:
# dim_data
dim_data = spark.sql("""
    SELECT DISTINCT
        DATE_FORMAT(SHIPPING_LIMIT_DATE, 'yyyyMMdd')  AS date_key,
        DATE(SHIPPING_LIMIT_DATE)                     AS date,
        YEAR(SHIPPING_LIMIT_DATE)                     AS year,
        MONTH(SHIPPING_LIMIT_DATE)                    AS month,
        DAY(SHIPPING_LIMIT_DATE)                      AS day,
        DATE_FORMAT(SHIPPING_LIMIT_DATE, 'EEEE')      AS weekday,
        QUARTER(SHIPPING_LIMIT_DATE)                  AS quarter
    FROM olist_silver
    WHERE SHIPPING_LIMIT_DATE IS NOT NULL
""")
dim_data.createOrReplaceTempView("dim_data")
print(f"dim_data: {dim_data.count()} datas distintas")
dim_data.show(5)

In [ ]:
# fct_order_items
fct_order_items = spark.sql("""
    SELECT
        CONCAT(ORDER_ID, '_', CAST(ORDER_ITEM_ID AS STRING))  AS order_item_key,
        ORDER_ID                                              AS order_id,
        MD5(PRODUCT_ID)                                       AS product_key,
        MD5(SELLER_ID)                                        AS seller_key,
        DATE_FORMAT(SHIPPING_LIMIT_DATE, 'yyyyMMdd')          AS date_key,
        PRICE                                                 AS price,
        FREIGHT_VALUE                                         AS freight_value
    FROM olist_silver
""")
fct_order_items.createOrReplaceTempView("fct_order_items")
print(f"fct_order_items: {fct_order_items.count()} itens")
fct_order_items.show(5)

## 6. Views Analíticas

As mesmas views definidas no Item 6 (`vw_receita_por_produto` e `vw_evolucao_temporal`), agora materializadas via Spark SQL.

In [ ]:
# View 1 — Receita por Produto
vw_receita_por_produto = spark.sql("""
    SELECT
        p.product_id,
        COUNT(DISTINCT f.order_id)                AS total_pedidos,
        COUNT(*)                                  AS total_itens,
        ROUND(SUM(f.price), 2)                    AS receita_total,
        ROUND(AVG(f.price), 2)                    AS preco_medio,
        ROUND(SUM(f.freight_value), 2)            AS frete_total,
        ROUND(SUM(f.price + f.freight_value), 2)  AS gmv_total
    FROM fct_order_items f
    JOIN dim_produto p ON f.product_key = p.product_key
    GROUP BY p.product_id
    ORDER BY receita_total DESC
""")

print("View 1 — vw_receita_por_produto (top 10):")
vw_receita_por_produto.show(10)

In [ ]:
# View 2 — Evolução Temporal
vw_evolucao_temporal = spark.sql("""
    SELECT
        d.year                                    AS ano,
        d.month                                   AS mes,
        COUNT(DISTINCT f.order_id)                AS total_pedidos,
        COUNT(*)                                  AS total_itens,
        ROUND(SUM(f.price), 2)                    AS receita_total,
        ROUND(AVG(f.price), 2)                    AS ticket_medio_item,
        ROUND(SUM(f.freight_value), 2)            AS frete_total
    FROM fct_order_items f
    JOIN dim_data d ON f.date_key = d.date_key
    GROUP BY d.year, d.month
    ORDER BY d.year, d.month
""")

print("View 2 — vw_evolucao_temporal:")
vw_evolucao_temporal.show(20)

## 7. Exportação dos resultados

In [ ]:
# Salva as views como CSV para evidência
vw_receita_por_produto.toPandas().to_csv("/content/vw_receita_por_produto.csv", index=False)
vw_evolucao_temporal.toPandas().to_csv("/content/vw_evolucao_temporal.csv", index=False)

# Salva as tabelas Gold como Parquet
fct_order_items.write.mode("overwrite").parquet("/content/gold/fct_order_items")
dim_produto.write.mode("overwrite").parquet("/content/gold/dim_produto")
dim_vendedor.write.mode("overwrite").parquet("/content/gold/dim_vendedor")
dim_data.write.mode("overwrite").parquet("/content/gold/dim_data")

print("Exportação concluída.")

In [ ]:
spark.stop()
print("SparkSession encerrada.")